# 1. Create HiSeq Object and initialize instruments. 

You can skip initializing the Cameras if you're only pumping/priming reagents.

In [1]:
import pyseq
import threading

hs = pyseq.HiSeq() # Create HiSeq Object
# hs.initializeCams() # Initialize Cameras
hs.initializeInstruments() # Initialize all other instruments (stages, pumps, valve, etc.)

2026-05-18 12:12 - HiSeq::Initializing FPGA 
2026-05-18 12:12 - HiSeq::Initializing X & Y stages 
2026-05-18 12:13 - HiSeq::Initializing lasers 
2026-05-18 12:16 - HiSeq::Initializing pumps and valves 
2026-05-18 12:16 - HiSeq::Initializing optics and Z stages 
2026-05-18 12:18 - HiSeq::Syncing Y stage 
2026-05-18 12:18 - HiSeq::Initialized!


True

# 2.  Configure instrument

Configure valves, pumps and then move the stage out

Default `waste_port` = 1, `flowcells` = A and B

In [2]:
# port_dict = {'PBS': 1} # Map reagents to valve ports

inlets = 2 # Use 2 port stage inlet, can 2 or 8. If 8 need a HiSeq 2000 or X flow cell

# Comment out the configuration you DONT use
flowcells = {'A':None, 'B':None}
# flowcells = {'B':None}
# flowcells = {'A':None}

hs.move_inlet(inlets) # Move to 2 port stage inlet
for fc in flowcells.keys():
    # loop through flow cells
    hs.v24[fc].port_dict = port_dict # Update valves with port mapping
    hs.p[fc].update_limits(8) # Configure pumps to use 8 barrels per flow cell lane

# Move stage Out
hs.move_stage_out() 

2026-05-18 12:20 - HiSeq::Moving stage out 


# 3. Lock flow cells on to the stage

# 4.  Load Reagents

**Load reagents into chiller, port 20, or on the left side of the sequencer**

# Note on defining priming parameters

| Location | Priming Volume uL|
|----------|------------------|
|Chiller   | 500              |
|Port 20   | 250              |
|Outside   | 350              |

Specify the volume for each reagent in a dictionary as follows
```{"PBS": 500}``` #volume in uL


# 5. Prime Reagents

Pumps volume specified in `vol_dict` from reagent to waste

Defaults: `flowrate` = 100 uL/min

In [7]:
# Volume to prime each reagent 
# vol_dict = {"PBS": 350}
vol_dict = {
            "PBS": 350,
            "water": 350,
           } 
flowrate = 100 # uL/min, reagents -> waste,  stay between 100 and 500 uL/min

# Move to waste_port
print(f'Moving to waste port: {waste_port}')
for fc in flowcells.keys():
    hs.v24[fc].move('waste')

# Loop through reagents
for reagent, vol in vol_dict.items():
    for fc in flowcells.keys():
        if reagent in hs.v24[fc].port_dict:
            print(f"Priming {vol} uL of {reagent}")
            # Move to reagent on valve
            hs.v24[fc].move(reagent)
            # Prime reagent
            flowcells[fc] = threading.Thread(target = hs.p[fc].pump, args  = (vol, flowrate))
            flowcells[fc].start()
        else:
            print(f"{reagent} not found on flowcell {fc}")
            flowcells[fc] = None

    # Wait for pumping to finish
    for fc in flowcells.keys():
        if flowcells[fc] is not None: 
            flowcells[fc].join()


Moving to waste port: 9
Priming 2000 of 2000


# 6. Pump a reagent through the flow cell

Specify `reagent` and `vol`

In [ ]:
reagent = "PBS"
vol = 500 # uL

for fc in flowcells.keys():
    if reagent in hs.v24[fc].port_dict:
        print(f"Priming {vol} uL of {reagent}")
        # Move to reagent on valve
        hs.v24[fc].move(reagent)
        # Prime reagent
        flowcells[fc] = threading.Thread(target = hs.p[fc].pump, args  = (vol, flowrate))
        flowcells[fc].start()
    else:
        print(f"{reagent} not found on flowcell {fc}")
        flowcells[fc] = None

# Wait for pumping to finish
for fc in flowcells.keys():
    if flowcells[fc] is not None: 
        flowcells[fc].join()

# 6. Reset
Moves stages & filters to safe position, turns off lasers

In [6]:
print("Reseting X-Stage")
hs.x.move(30000) # center x stage
print("Reseting Y-Stage")
hs.y.move(0) # center y stage,
hs.y.command("OFF") # turn off y stage
print("Reseting Z-Stage")
hs.z.move([0, 0, 0]) # lower tilt stage
print("Reseting Lasers and Filters")
for color in ["red", "green"]:
    hs.optics.move_ex(color, 'home') # home emission filters
    hs.lasers[color].turn_on(False) # Turn off lasers
print("Reset complete")

## If using HiSeq within 10 days after reset, leave the system on.
1. Leave flow cells locked on to the stage and submerge all line inlets in water to prevent them from drying out.

2. Close the stage door.

3. Idle the sequencer: ```pyseq -idle```

## If not using HiSeq in more than 10 day after reset, turn off the sytem.
1. Remove the flow cells from the stage and all reagents from the chiller

2. Close the stage door

3. Turn the power off to the HiSeq